# Automotive supply chain — two-week capstone plan

Single runnable pipeline aligned to the 2-week plan: extract Moetz et al. (2020) workbook → profile → (later) Neo4j, EDA, scenarios, report.

**Definition of done (business):** quantify how a supply-side shock propagates through the BOM and network; locate periods/arcs where stress concentrates; compare at least one mitigation vs baseline.

**Docs:** [docs/day1/research_questions.md](docs/day1/research_questions.md) · [docs/day1/runbook.md](docs/day1/runbook.md)

**Legacy reference (do not delete):** `AutomotiveSupplyChain_Merged_Submission.ipynb` — lift additional Cypher/GDS cells from there when you reach Week 2.

---

| Block | Days | Focus |
|-------|------|--------|
| Week 1 | 1–5 | Scope (docs), extraction, profiling, DGP notes, paper ↔ column mapping |
| Week 2 | 6–10 | Neo4j load, EDA, bottlenecks, scenarios, final writeup |


## Setup — paths and imports

Run Jupyter with **working directory = `finalProject`** (same as [docs/day1/runbook.md](docs/day1/runbook.md)).

In [5]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name != "finalProject" and (ROOT / "2020_dataset_OfAutomotiveProductionNetwork.xlsb").is_file():
    pass
elif (ROOT / "finalProject" / "2020_dataset_OfAutomotiveProductionNetwork.xlsb").is_file():
    ROOT = ROOT / "finalProject"

WORKBOOK = ROOT / "2020_dataset_OfAutomotiveProductionNetwork.xlsb"
EXPORT_DIR = ROOT / "data_export"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DAY2 = ROOT / "docs" / "day2"
DOCS_DAY2.mkdir(parents=True, exist_ok=True)
DOCS_DAY3 = ROOT / "docs" / "day3"
DOCS_DAY3.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("ROOT:", ROOT)
print("WORKBOOK exists:", WORKBOOK.is_file(), WORKBOOK)
print("EXPORT_DIR:", EXPORT_DIR)
print("DOCS_DAY3:", DOCS_DAY3)

Python: 3.13.9
ROOT: /home/themad/Documents/yeshiva/capstone/neo4j/finalProject
WORKBOOK exists: True /home/themad/Documents/yeshiva/capstone/neo4j/finalProject/2020_dataset_OfAutomotiveProductionNetwork.xlsb
EXPORT_DIR: /home/themad/Documents/yeshiva/capstone/neo4j/finalProject/data_export
DOCS_DAY3: /home/themad/Documents/yeshiva/capstone/neo4j/finalProject/docs/day3


## Week 1 — Day 2: Extract workbook tabs → CSV (`pyxlsb`)

Expected sheets match the Mendeley dataset; filenames drive downstream `LOAD CSV` and profiling.

In [6]:
EXPECTED_SHEETS = [
    "products",
    "nodes",
    "nodes_inflow",
    "arcs",
    "capacity_at_arc",
    "max_flow_product_per_arc",
    "max_flow_group_per_arc",
    "operations",
    "BOM",
    "demands",
    "initial_inventories",
    "initial_flows",
]

if not WORKBOOK.is_file():
    print(
        "MISSING workbook. Download from Mendeley Data and save as:\n",
        WORKBOOK.name,
    )
else:
    xl = pd.ExcelFile(WORKBOOK, engine="pyxlsb")
    available = set(xl.sheet_names)
    print("Sheets in file:", len(available))
    exported = []
    missing = []
    for name in EXPECTED_SHEETS:
        sheet_name = name
        if sheet_name not in available:
            lower = {s.lower(): s for s in available}
            key = name.lower()
            if key in lower:
                sheet_name = lower[key]
            else:
                missing.append(name)
                continue
        df = pd.read_excel(WORKBOOK, sheet_name=sheet_name, engine="pyxlsb")
        out = EXPORT_DIR / f"{name}.csv"
        df.to_csv(out, index=False, encoding="utf-8")
        exported.append((name, len(df), len(df.columns), list(df.columns)))
    for row in exported:
        print(f"  {row[0]}: {row[1]} rows × {row[2]} cols -> {row[0]}.csv")
    if missing:
        print("NOT FOUND (check exact sheet names in workbook):", missing)

Sheets in file: 12
  products: 28049 rows × 3 cols -> products.csv
  nodes: 12 rows × 1 cols -> nodes.csv
  nodes_inflow: 44 rows × 2 cols -> nodes_inflow.csv
  arcs: 11 rows × 4 cols -> arcs.csv
  capacity_at_arc: 154 rows × 4 cols -> capacity_at_arc.csv
  max_flow_product_per_arc: 365 rows × 5 cols -> max_flow_product_per_arc.csv
  max_flow_group_per_arc: 20 rows × 5 cols -> max_flow_group_per_arc.csv
  operations: 15 rows × 7 cols -> operations.csv
  BOM: 87059 rows × 3 cols -> BOM.csv
  demands: 28000 rows × 4 cols -> demands.csv
  initial_inventories: 82 rows × 6 cols -> initial_inventories.csv
  initial_flows: 117 rows × 5 cols -> initial_flows.csv


## Week 1 — Day 2: Write `extraction_manifest.md`

Committed snapshot for audit (CSV blobs stay gitignored under `data_export/`).

In [7]:
from datetime import datetime, timezone

try:
    import pyxlsb

    pyxlsb_ver = getattr(pyxlsb, "__version__", "unknown")
except Exception:
    pyxlsb_ver = "n/a"

manifest_path = DOCS_DAY2 / "extraction_manifest.md"
lines = [
    "# Extraction manifest",
    "",
    "- **Generated (UTC):** " + datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    f"- **Python:** {sys.version.split()[0]}",
    f"- **pyxlsb:** {pyxlsb_ver}",
    f"- **Workbook:** `{WORKBOOK.name}`",
    "",
    "| Sheet key | Rows | Cols | Columns |",
    "|-----------|------|------|---------|",
]
csvs = sorted(EXPORT_DIR.glob("*.csv"))
for p in csvs:
    full = pd.read_csv(p, low_memory=False)
    cols = ", ".join(full.columns.astype(str))
    lines.append(f"| {p.stem} | {len(full):,} | {full.shape[1]} | {cols} |")
lines.append("")
manifest_path.write_text("\n".join(lines), encoding="utf-8")
print("Wrote", manifest_path)

Wrote /home/themad/Documents/yeshiva/capstone/neo4j/finalProject/docs/day2/extraction_manifest.md


## Week 1 — Day 3: Full profile + cross-table integrity

Runs full-row profiling (nulls, distincts, `period_t` ranges) and referential checks vs `nodes` / `products`, then writes [`docs/day3/profile_integrity_summary.md`](docs/day3/profile_integrity_summary.md). Logic matches `AutomotiveSupplyChain_Merged_Submission.ipynb` §4.

In [8]:
from datetime import datetime, timezone

# --- Full-file profile (same as Merged_Submission §4) ---
csvs = sorted(EXPORT_DIR.glob("*.csv"))
report_lines: list[str] = [
    "# Profile and integrity summary (Day 3)",
    "",
    f"- **Generated (UTC):** {datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')}",
    "",
]

if not csvs:
    print("No CSVs in", EXPORT_DIR, "— run Day 2 extraction first.")
    report_lines.append("_No CSVs; run Day 2 first._")
else:
    report_lines.extend(["## Full-file profile", ""])
    for p in csvs:
        df = pd.read_csv(p, low_memory=False)
        na = df.isna().sum()
        na_nonzero = na[na > 0]
        report_lines.extend(
            [
                f"### {p.name}",
                f"rows={len(df):,}  cols={df.shape[1]}",
                "",
                df.dtypes.to_string(),
                "",
            ]
        )
        if len(na_nonzero):
            report_lines.append("Null counts (nonzero):")
            report_lines.append(na_nonzero.to_string())
        else:
            report_lines.append("Null counts: none")
        report_lines.extend(["", "Distinct counts:"])
        for col in df.columns:
            report_lines.append(f"  {col}: {df[col].nunique():,}")
        if "period_t" in df.columns:
            report_lines.append(
                f"  period_t range: {df['period_t'].min()} … {df['period_t'].max()}"
            )
        report_lines.append("")
        print(f"profiled {p.name}: {len(df):,} rows")

    # --- Cross-table integrity ---
    paths = {p.stem: p for p in csvs}

    def _load(stem: str) -> pd.DataFrame:
        return pd.read_csv(paths[stem], low_memory=False)

    def _numeric_ids(s: pd.Series) -> set[int]:
        v = pd.to_numeric(s, errors="coerce").dropna()
        return set(v.astype(int))

    issues: list[str] = []

    if "nodes" not in paths:
        issues.append("SKIP: nodes.csv missing")
    else:
        nodes_df = _load("nodes")
        node_set = set(nodes_df["node_n"].dropna().astype(str))

        def nodes_ref_ok(stem: str, cols: list[str]) -> None:
            if stem not in paths:
                return
            df = _load(stem)
            for col in cols:
                if col not in df.columns:
                    continue
                vals = set(df[col].dropna().astype(str))
                unknown = vals - node_set
                if unknown:
                    sample = sorted(unknown)[:8]
                    issues.append(
                        f"WARN {stem}.{col}: {len(unknown)} values not in nodes.node_n e.g. {sample}"
                    )

        nodes_ref_ok("arcs", ["starting_node_i", "ending_node_j"])
        nodes_ref_ok("demands", ["node_n"])
        nodes_ref_ok("initial_inventories", ["node_n"])
        nodes_ref_ok("operations", ["node_n"])
        nodes_ref_ok("nodes_inflow", ["node_n"])
        for stem in (
            "capacity_at_arc",
            "max_flow_group_per_arc",
            "max_flow_product_per_arc",
            "initial_flows",
        ):
            nodes_ref_ok(stem, ["starting_node_i", "ending_node_j"])

    if "products" not in paths:
        issues.append("SKIP: products.csv missing")
    elif "BOM" in paths:
        products_df = _load("products")
        bom_df = _load("BOM")
        prod_int = _numeric_ids(products_df["product_p"])
        mothers = _numeric_ids(bom_df["mother"])
        missing_mothers = mothers - prod_int
        if missing_mothers:
            issues.append(
                f"WARN BOM.mother: {len(missing_mothers)} mothers not in products.product_p e.g. {sorted(missing_mothers)[:8]}"
            )
        prod_str = {str(int(x)) for x in prod_int}
        children = set(bom_df["child"].dropna().astype(str))
        overlap = children & prod_str
        issues.append(
            f"INFO BOM.child: {len(children)} distinct child codes; {len(overlap)} appear as stringified products.product_p (rest are part codes only in BOM)"
        )

    if "demands" in paths and "products" in paths:
        ddf = _load("demands")
        pdf = _load("products")
        pset = _numeric_ids(pdf["product_p"])
        dprods = _numeric_ids(ddf["product_p"])
        missing_d = dprods - pset
        if missing_d:
            issues.append(
                f"WARN demands.product_p: {len(missing_d)} not in products e.g. {sorted(missing_d)[:8]}"
            )

    if "initial_inventories" in paths and "products" in paths:
        inv = _load("initial_inventories")
        pdf = _load("products")
        prod_str = {str(x) for x in _numeric_ids(pdf["product_p"])}
        inv_p = set(inv["product_p"].dropna().astype(str))
        overlap_inv = inv_p & prod_str
        issues.append(
            f"INFO initial_inventories.product_p: {len(inv_p)} distinct symbols; {len(overlap_inv)} match stringified numeric products (others e.g. BEV-style codes — still model as :Product)"
        )

    report_lines.extend(["## Cross-table integrity", ""])
    if not issues:
        report_lines.append("PASS: no WARN/SKIP lines (INFO lines may still apply).")
        print("PASS: no referential WARN/SKIP (see INFO lines in report).")
    else:
        report_lines.extend(issues)
        for line in issues:
            print(line)

summary_path = DOCS_DAY3 / "profile_integrity_summary.md"
summary_path.write_text("\n".join(report_lines), encoding="utf-8")
print("Wrote", summary_path)

profiled BOM.csv: 87,059 rows
profiled arcs.csv: 11 rows
profiled capacity_at_arc.csv: 154 rows
profiled demands.csv: 28,000 rows
profiled initial_flows.csv: 117 rows
profiled initial_inventories.csv: 82 rows
profiled max_flow_group_per_arc.csv: 20 rows
profiled max_flow_product_per_arc.csv: 365 rows
profiled nodes.csv: 12 rows
profiled nodes_inflow.csv: 44 rows
profiled operations.csv: 15 rows
profiled products.csv: 28,049 rows
INFO BOM.child: 49 distinct child codes; 0 appear as stringified products.product_p (rest are part codes only in BOM)
INFO initial_inventories.product_p: 45 distinct symbols; 0 match stringified numeric products (others e.g. BEV-style codes — still model as :Product)
Wrote /home/themad/Documents/yeshiva/capstone/neo4j/finalProject/docs/day3/profile_integrity_summary.md


## Week 1 — Day 4: DGP and data reliability (complete)

**Committed write-up:** [docs/day4/data_reliability_note.md](docs/day4/data_reliability_note.md) · [docs/day4/README.md](docs/day4/README.md)

**Framework:** [DGP_EDA_Beginner_Guide.md](DGP_EDA_Beginner_Guide.md). **Evidence:** [docs/day3/profile_integrity_summary.md](docs/day3/profile_integrity_summary.md).

**Reader summary**

- DGP mixes industry-style structure with simulated/completed series (see paper §4.7); claims are about **this tactical instance**, not universal OEM ERP truth.
- Day 3 shows **no null cells** in exports; “complete tables” still omit real-world factors (latent missingness).
- **Period alignment:** planning windows (`demands` / `capacity_at_arc` 61–74) vs history (`initial_flows` 39–60) vs snapshot (`initial_inventories` at 60)—interpret together, not as errors.
- **BOM vs `products`:** child part codes need not be product-catalog rows (Day 3 INFO); model parts explicitly.
- **KPIs:** propagation and bottlenecks are strong **within-network**; pair centrality with capacity/lead time before managerial conclusions.

---

## Week 1 — Day 5: Paper symbol mapping (complete)

**Committed:** [docs/day5/paper_symbol_mapping.md](docs/day5/paper_symbol_mapping.md) · [docs/day5/README.md](docs/day5/README.md)

**Use in Week 2**

- **Day 6 Neo4j:** Align relationship properties (e.g. `l_ij`, `c_ijt`, `d_npt`, operation flags) with this table when copying `LOAD CSV` from `AutomotiveSupplyChain_Merged_Submission.ipynb`.
- **Report methods:** Use the same symbols in text so equations in the paper match your column names.
- **Not in this doc:** Concrete Neo4j *labels* and *relationship types* — hardened on Day 6; this file is **semantics**, not the full graph DDL.

## Week 2 — Days 6–10 (placeholders)

1. Neo4j: constraints, `LOAD CSV` order, `run_query` helper — copy from `AutomotiveSupplyChain_Merged_Submission.ipynb` §6–7.
2. EDA: facility graph, BOM depth, lead-time paths.
3. Capacity vs demand and bottleneck tables.
4. Scenario: baseline vs shortage vs mitigation; KPI deltas.
5. Export figures and finalize report narrative.